# Crash Course — Transcriptomics (전사체)

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [crash-course/transcriptomics.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/crash-course/transcriptomics.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 최소 실행 예시

### 예시 1 — raw count와 log 변환값의 PCA 비교

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

rng = np.random.default_rng(1)
n_genes, n_per_group = 2000, 6
samples = [f"ctrl_{i+1}" for i in range(n_per_group)] + [f"treat_{i+1}" for i in range(n_per_group)]

# 가상 raw count: 유전자마다 기본 발현량이 다르고, 앞 200개 유전자는 처리군에서 3배
base = rng.lognormal(mean=4, sigma=1.5, size=n_genes)
fold = np.ones((n_genes, 2 * n_per_group))
fold[:200, n_per_group:] = 3
depth = np.array([1.0, 0.6, 1.4, 0.9, 1.2, 0.7, 1.1, 0.5, 2.0, 0.8, 1.3, 1.6])   # 샘플마다 다른 시퀀싱 깊이
mu = base[:, None] * fold * depth[None, :]
counts = pd.DataFrame(rng.negative_binomial(n=10, p=10 / (10 + mu)),
                      index=[f"gene_{i+1}" for i in range(n_genes)], columns=samples)

print("샘플별 총 read 수 (library size)")
print(counts.sum().to_string())
print()

def pca_report(matrix, label):
    # 행 = 샘플이 되도록 뒤집고, 유전자별 평균을 뺀 뒤 PCA
    x = matrix.T.values
    pca = PCA(n_components=2)
    pcs = pca.fit_transform(x - x.mean(axis=0))
    ratio = pca.explained_variance_ratio_
    print(f"[{label}] PC1 {ratio[0]:.0%}, PC2 {ratio[1]:.0%}")
    out = pd.DataFrame(pcs, index=matrix.columns, columns=["PC1", "PC2"]).round(1)
    out["lib_size_M"] = (counts.sum() / 1e6).round(2)
    print(out.to_string())
    r = np.corrcoef(pcs[:, 0], counts.sum())[0, 1]
    print(f"PC1과 library size의 상관: {r:.2f}")
    print()
    return pcs

# (가) raw count 그대로 PCA — 하면 안 되는 방식
pca_report(counts, "raw count")

# (나) CPM 정규화 후 log2(CPM + 1) — 탐색·시각화용 값
keep = (counts >= 10).sum(axis=1) >= n_per_group          # 낮은 발현 유전자 제거
cpm = counts[keep] / counts[keep].sum() * 1e6
logcpm = np.log2(cpm + 1)
print("필터 후 유전자 수:", int(keep.sum()))
pcs = pca_report(logcpm, "log2(CPM+1)")

# 자기 점검: 정규화 후에는 PC1이 군을 완전히 가른다
is_treat = np.array([s.startswith("treat") for s in samples])
assert pcs[is_treat, 0].min() > pcs[~is_treat, 0].max() or pcs[is_treat, 0].max() < pcs[~is_treat, 0].min()

### 예시 2 — 유전자별 검정과 BH 보정

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

rng = np.random.default_rng(1)
n_genes, n = 2000, 6
base = rng.lognormal(mean=4, sigma=1.5, size=n_genes)
fold = np.ones((n_genes, 2 * n)); fold[:200, n:] = 3        # 진짜 변한 유전자: 앞 200개
depth = np.array([1.0, 0.6, 1.4, 0.9, 1.2, 0.7, 1.1, 0.5, 2.0, 0.8, 1.3, 1.6])
mu = base[:, None] * fold * depth[None, :]
counts = pd.DataFrame(rng.negative_binomial(n=10, p=10 / (10 + mu)),
                      index=[f"gene_{i+1}" for i in range(n_genes)])
truth = pd.Series(np.arange(n_genes) < 200, index=counts.index)

keep = (counts >= 10).sum(axis=1) >= n
logcpm = np.log2(counts[keep] / counts[keep].sum() * 1e6 + 1)
ctrl, treat = logcpm.iloc[:, :n], logcpm.iloc[:, n:]

# 유전자마다 같은 검정을 반복한다 (Welch t-test: 학습용 대역. 실제 분석은 DESeq2/edgeR/limma)
t, p = stats.ttest_ind(treat, ctrl, axis=1, equal_var=False)
res = pd.DataFrame({"log2FC": treat.mean(axis=1) - ctrl.mean(axis=1), "p": p}, index=logcpm.index)
res["padj"] = multipletests(res["p"], method="fdr_bh")[1]
res["truth"] = truth[res.index]

print("검정한 유전자 수:", len(res), "/ 그중 진짜 변한 유전자:", int(res["truth"].sum()))
for name, hit in [("raw p < 0.05", res["p"] < 0.05),
                  ("BH padj < 0.05", res["padj"] < 0.05),
                  ("Bonferroni < 0.05", res["p"] * len(res) < 0.05)]:
    tp = int((hit & res["truth"]).sum()); fp = int((hit & ~res["truth"]).sum())
    print(f"{name:18s} 선택 {hit.sum():4d}개 | 진짜 {tp:3d} | 가짜 {fp:3d}")
print()
print(res.sort_values("p").head(5).to_string(float_format=lambda v: f"{v:.3g}"))

assert (res["padj"] >= res["p"]).all()      # 보정값은 원래 p보다 작아질 수 없다